In [27]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

# -
from src.preprocessing.event_index import compute_event_index
from src.data.evimo2.parser import EVIMO2Parser




Project Root: /home/ayon/git/EventCameraProject


In [28]:
from src.preprocessing.camera_motion import (
    generate_camera_motion,
    load_camera_motion,
)
sequence_dir = Path("/home/ayon/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000")


generate_camera_motion(sequence_dir)

cache = load_camera_motion(sequence_dir)

print("Frames:", len(cache.frame_ids))
print("Valid poses:", cache.pose_available.sum())
print("Missing poses:", (~cache.pose_available).sum())

print()
print("Translation:", cache.translation.shape)
print("Quaternion :", cache.quaternion.shape)

print()
print("Delta Translation:", cache.delta_translation.shape)
print("Linear Speed     :", cache.linear_speed.shape)

print()
print("Delta Quaternion :", cache.delta_quaternion.shape)
print("Angular Speed    :", cache.angular_speed.shape)

print()
print("Maximum linear speed :", cache.linear_speed.max())
print("Maximum angular speed:", cache.angular_speed.max())

Frames: 404
Valid poses: 392
Missing poses: 12

Translation: (404, 3)
Quaternion : (404, 4)

Delta Translation: (404, 3)
Linear Speed     : (404,)

Delta Quaternion : (404, 4)
Angular Speed    : (404,)

Maximum linear speed : 0.5250032
Maximum angular speed: 0.86013615


In [29]:
import numpy as np

print("Pose available dtype:", cache.pose_available.dtype)

print("Missing pose indices:")
print(np.where(~cache.pose_available)[0])

print()

assert np.all(cache.linear_speed[~cache.pose_available] == 0)
print("✓ Linear speed zero for missing poses")

assert np.all(cache.angular_speed[~cache.pose_available] == 0)
print("✓ Angular speed zero for missing poses")

assert np.allclose(
    cache.delta_translation[~cache.pose_available],
    0,
)
print("✓ Delta translation zero")

assert np.allclose(
    cache.delta_quaternion[~cache.pose_available],
    np.array([0, 0, 0, 1], dtype=np.float32),
)
print("✓ Identity quaternion for missing poses")

Pose available dtype: bool
Missing pose indices:
[ 61  62  63  64  65  66  67  68  69  70 289 290]

✓ Linear speed zero for missing poses
✓ Angular speed zero for missing poses
✓ Delta translation zero
✓ Identity quaternion for missing poses


In [30]:
missing = np.where(~cache.pose_available)[0]

print("Missing:", missing)

for i in [60, 61, 70, 71, 288, 289, 290, 291]:
    print(
        i,
        "pose:", cache.pose_available[i],
        "linear:", cache.linear_speed[i],
        "angular:", cache.angular_speed[i],
    )
    

Missing: [ 61  62  63  64  65  66  67  68  69  70 289 290]
60 pose: True linear: 0.17703554 angular: 0.62559825
61 pose: False linear: 0.0 angular: 0.0
70 pose: False linear: 0.0 angular: 0.0
71 pose: True linear: 0.11051712 angular: 0.45992416
288 pose: True linear: 0.16220018 angular: 0.6032481
289 pose: False linear: 0.0 angular: 0.0
290 pose: False linear: 0.0 angular: 0.0
291 pose: True linear: 0.17024817 angular: 0.5813675


In [31]:
sequence_dir = Path("/home/ayon/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000")
parser = EVIMO2Parser(sequence_dir)
print(type(parser))

print("\nAttributes:")
for x in sorted(dir(parser)):
    if not x.startswith("_"):
        print(x)

frame = parser.frames[0]

print(type(frame))

from dataclasses import asdict

for k, v in asdict(frame).items():
    print(k)
    print(type(v))
    print(v)
    print("-"*60)


obj = parser.frames[0].object_states[0]

print(type(obj))

try:
    print(obj.__dict__)
except:
    from dataclasses import asdict
    print(asdict(obj))

<class 'src.data.evimo2.parser.EVIMO2Parser'>

Attributes:
frames
get_events
get_frame
imu
load_depth
load_mask
objects
reader
sequence
sequence_dir
<class 'src.data.evimo2.datatypes.raw.RawFrame'>
frame_id
<class 'int'>
0
------------------------------------------------------------
timestamp
<class 'float'>
0.016667
------------------------------------------------------------
camera_pose
<class 'dict'>
{'translation': array([ 0.002478,  0.000536, -0.000898], dtype=float32), 'quaternion': array([-2.46000e-03,  9.81000e-04, -4.07200e-03,  9.99988e-01],
      dtype=float32)}
------------------------------------------------------------
object_states
<class 'tuple'>
({'object_id': 11, 'pose': {'translation': array([-0.504081, -0.534931,  1.169943], dtype=float32), 'quaternion': array([-0.243625,  0.228212, -0.76369 ,  0.552579], dtype=float32)}, 'visible': True}, {'object_id': 14, 'pose': {'translation': array([-0.550447, -0.783066,  1.253755], dtype=float32), 'quaternion': array([-0.17487

In [32]:
seq = parser.sequence

print(type(seq))

try:
    print(seq.__dict__)
except:
    from dataclasses import asdict
    print(asdict(seq))

<class 'src.data.evimo2.datatypes.raw.RawSequence'>
{'sequence_name': 'scene13_dyn_test_01_000000', 'camera': {'fx': 556.573975, 'fy': 556.086975, 'cx': 321.846008, 'cy': 225.617996, 'width': 640, 'height': 480, 'distortion': array([-0.10828 ,  0.206233,  0.      ,  0.      ], dtype=float32)}, 'num_events': 38328869, 'num_frames': 404, 'start_time': 0.016667, 'end_time': 6.733333, 'objects': {11: {'object_id': 11, 'name': 'object_11'}, 14: {'object_id': 14, 'name': 'object_14'}, 15: {'object_id': 15, 'name': 'object_15'}, 22: {'object_id': 22, 'name': 'object_22'}, 23: {'object_id': 23, 'name': 'object_23'}, 24: {'object_id': 24, 'name': 'object_24'}, 5: {'object_id': 5, 'name': 'object_5'}, 6: {'object_id': 6, 'name': 'object_6'}}}


In [33]:
import numpy as np

info = np.load(
    sequence_dir / "dataset_info.npz",
    allow_pickle=True,
)

print(info.files)

for key in info.files:
    print("="*60)
    print(key)
    print(type(info[key]))
    print(info[key].shape)

['index', 'discretization', 'K', 'D', 'meta']
index
<class 'numpy.ndarray'>
(670,)
discretization
<class 'numpy.ndarray'>
()
K
<class 'numpy.ndarray'>
(3, 3)
D
<class 'numpy.ndarray'>
(4,)
meta
<class 'numpy.ndarray'>
()


In [34]:
frame = parser.frames[0]

for name in dir(frame):
    if any(k in name.lower() for k in [
        "pose",
        "camera",
        "rotation",
        "translation",
        "transform",
        "world",
        "quat",
        "matrix",
        "t_",
    ]):
        print(name)

__format__
__gt__
__init__
__init_subclass__
__lt__
camera_pose
object_states


In [35]:
sequence_dir = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000"
)

dataset = EVIMO2Parser(sequence_dir)

cache = compute_event_index(dataset)

print(cache.keys())

print()

for k, v in cache.items():
    print(
        f"{k:18}",
        v.shape,
        v.dtype,
    )

print(cache["event_start"][:5])
print(cache["event_end"][:5])
print(cache["event_count"][:5])

print()

print("Depth GT :", cache["depth_available"].sum())
print("Mask GT  :", cache["mask_available"].sum())

dict_keys(['frame_ids', 'timestamps', 'frame_dt', 'event_start', 'event_end', 'event_count', 'has_events', 'depth_available', 'mask_available'])

frame_ids          (404,) int32
timestamps         (404,) float64
frame_dt           (404,) float64
event_start        (404,) int64
event_end          (404,) int64
event_count        (404,) int64
has_events         (404,) bool
depth_available    (404,) bool
mask_available     (404,) bool
[     0  98954 196243 291980 388218]
[ 98954 196243 291980 388218 487013]
[98954 97289 95737 96238 98795]

Depth GT : 392
Mask GT  : 392


In [36]:
from src.data.evimo2.parser import EVIMO2Parser
from src.preprocessing.frame_motion import compute_frame_motion

dataset = EVIMO2Parser(
    sequence_dir
)

cache = compute_frame_motion(
    dataset.frames,
    dataset.objects,
)

print(cache.frame_ids.shape)
print(cache.object_ids)
print(cache.delta_position.shape)
print(cache.speed.shape)

print(cache.speed.max())

(404,)
[ 5  6 11 14 15 22 23 24]
(404, 8, 3)
(404, 8)
1.0280248


In [37]:
from pathlib import Path
from src.data.evimo2.parser import EVIMO2Parser

sequence = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/"
    "flea3_7/sanity/tabletop/tabletop_2_flat_lr_000000"
)

dataset = EVIMO2Parser(sequence)

print("Objects listed in dataset_info:")
print(sorted(dataset.objects.keys()))

print()

frame_object_ids = set()

for frame in dataset.frames:
    for state in frame.object_states:
        frame_object_ids.add(state.object_id)

print("Objects actually appearing in frames:")
print(sorted(frame_object_ids))

print()

print("Missing from dataset.objects:")
print(sorted(frame_object_ids - set(dataset.objects.keys())))

Objects listed in dataset_info:
[22]

Objects actually appearing in frames:
[22]

Missing from dataset.objects:
[]


In [38]:
from pathlib import Path
from src.data.evimo2.parser import EVIMO2Parser

sequence = Path(
"/home/ayon/HDD/EventDatasets/EVIMO2_official/flea3_7/sanity/tabletop/tabletop_2_flat_lr_000000"
)

dataset = EVIMO2Parser(sequence)

print("dataset.objects")
print(sorted(dataset.objects.keys()))

print()

print("Scanning frames...")

all_ids = set()

for frame in dataset.frames:
    all_ids.update(s.object_id for s in frame.object_states)

print("IDs appearing in frames:")
print(sorted(all_ids))

print()

print("Missing IDs:")
print(sorted(all_ids - set(dataset.objects.keys())))

dataset.objects
[22]

Scanning frames...
IDs appearing in frames:
[22]

Missing IDs:
[]


In [39]:
from pathlib import Path
from src.data.evimo2.parser import EVIMO2Parser

sequence_dir = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/"
    "flea3_7/sanity/tabletop/tabletop_2_flat_lr_000000"
)

dataset = EVIMO2Parser(sequence_dir)

print("Objects declared in dataset.objects:")
print(sorted(dataset.objects.keys()))
print()

all_ids = set()

for frame in dataset.frames:
    for state in frame.object_states:
        all_ids.add(state.object_id)

print("Objects appearing in frames:")
print(sorted(all_ids))
print()

print("Missing from dataset.objects:")
print(sorted(all_ids - set(dataset.objects.keys())))
print()

print("Extra in dataset.objects:")
print(sorted(set(dataset.objects.keys()) - all_ids))

Objects declared in dataset.objects:
[22]

Objects appearing in frames:
[22]

Missing from dataset.objects:
[]

Extra in dataset.objects:
[]


In [40]:
frame = parser.frames[0]

pose = frame.camera_pose

print(type(pose))

print("\nDIR")
for x in dir(pose):
    if not x.startswith("_"):
        print(x)

print("\nDICT")
try:
    print(vars(pose))
except Exception as e:
    print(e)

print("\nASDICT")
from dataclasses import asdict
try:
    print(asdict(pose))
except Exception as e:
    print(e)

<class 'src.data.evimo2.datatypes.common.Pose'>

DIR
quaternion
translation

DICT
vars() argument must have __dict__ attribute

ASDICT
{'translation': array([ 0.002478,  0.000536, -0.000898], dtype=float32), 'quaternion': array([-2.46000e-03,  9.81000e-04, -4.07200e-03,  9.99988e-01],
      dtype=float32)}


In [41]:
obj = parser.frames[0].object_states[0]

pose = obj.pose

print(type(pose))

print("\nDIR")
for x in dir(pose):
    if not x.startswith("_"):
        print(x)

print("\nDICT")
try:
    print(vars(pose))
except Exception as e:
    print(e)

print("\nASDICT")
from dataclasses import asdict
try:
    print(asdict(pose))
except Exception as e:
    print(e)

<class 'src.data.evimo2.datatypes.common.Pose'>

DIR
quaternion
translation

DICT
vars() argument must have __dict__ attribute

ASDICT
{'translation': array([-0.504081, -0.534931,  1.169943], dtype=float32), 'quaternion': array([-0.243625,  0.228212, -0.76369 ,  0.552579], dtype=float32)}


In [42]:
from src.data.evimo2.datatypes.raw import Pose

print(Pose)

print("\nAnnotations")
print(Pose.__annotations__)

<class 'src.data.evimo2.datatypes.common.Pose'>

Annotations
{'translation': 'np.ndarray', 'quaternion': 'np.ndarray'}


In [43]:
from pathlib import Path
from src.data.evimo2.parser import EVIMO2Parser

sequence_dir = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/"
    "flea3_7/sfm/train/scene_04_00_000001"
    # "flea3_7/sanity/tabletop/tabletop_2_flat_lr_000000"
)

# flea3_7/sfm/train/scene_04_00_000001Traceback

dataset = EVIMO2Parser(sequence_dir)

print("Objects declared in dataset.objects:")
print(sorted(dataset.objects.keys()))
print()

all_ids = set()

for frame in dataset.frames:
    for state in frame.object_states:
        all_ids.add(state.object_id)

print("Objects appearing in frames:")
print(sorted(all_ids))
print()

print("Missing from dataset.objects:")
print(sorted(all_ids - set(dataset.objects.keys())))
print()

print("Extra in dataset.objects:")
print(sorted(set(dataset.objects.keys()) - all_ids))

Objects declared in dataset.objects:
[6, 8, 9, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26, 27]

Objects appearing in frames:
[6, 8, 9, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26, 27]

Missing from dataset.objects:
[]

Extra in dataset.objects:
[]


In [44]:
from pathlib import Path
from src.data.evimo2.parser import EVIMO2Parser

sequence_dir = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/"
    "flea3_7/sfm/train/scene_04_00_000001"
)

dataset = EVIMO2Parser(sequence_dir)

trajectory_data = {
    object_id: {
        "timestamps": [],
        "positions": [],
    }
    for object_id in dataset.objects
}

print(type(dataset.objects))
print("trajectory_data keys:")
print(sorted(trajectory_data.keys()))

<class 'dict'>
trajectory_data keys:
[6, 8, 9, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26, 27]


In [45]:
for frame in dataset.frames:
    for state in frame.object_states:
        if state.object_id not in dataset.objects:
            print(
                "Frame:",
                frame.frame_id,
                "Timestamp:",
                frame.timestamp,
                "Object:",
                state.object_id,
            )

In [46]:
from pathlib import Path
from src.data.evimo2.parser import EVIMO2Parser

sequence_dir = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/"
    "flea3_7/sfm/train/scene_04_00_000001"
)

dataset = EVIMO2Parser(sequence_dir)

trajectory_data = {
    object_id: {
        "timestamps": [],
        "positions": [],
    }
    for object_id in dataset.objects
}

print(type(dataset.objects))
print("trajectory_data keys:")
print(sorted(trajectory_data.keys()))


print(type(dataset.objects))
print(type(next(iter(dataset.objects.keys()))))
print(type(dataset.frames[0].object_states[0].object_id))


trajectory_data = {
    object_id: {
        "timestamps": [],
        "positions": [],
    }
    for object_id in dataset.objects
}

for frame in dataset.frames:
    if frame.camera_pose is None:
        continue

    for state in frame.object_states:
        if state.object_id not in trajectory_data:
            print("Missing key!")
            print("Frame:", frame.frame_id)
            print("Object:", state.object_id)
            print("Type:", type(state.object_id))
            break

<class 'dict'>
trajectory_data keys:
[6, 8, 9, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26, 27]
<class 'dict'>
<class 'int'>
<class 'int'>


In [47]:
from pathlib import Path

from src.data.evimo2.parser import EVIMO2Parser
from src.preprocessing.object_motion import collect_world_trajectories

sequence_dir = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/"
    "/flea3_7/sfm/eval/scene_03_02_000002"
)

dataset = EVIMO2Parser(sequence_dir)

In [48]:
import numpy as np

from src.preprocessing.object_motion import (
    ObjectTrajectory,
    pose_to_matrix,
)

def debug_collect_world_trajectories(frames, objects):

    trajectory_data = {
        object_id: {
            "timestamps": [],
            "positions": [],
        }
        for object_id in objects
    }

    print("Initial keys:")
    print(sorted(trajectory_data.keys()))
    print()

    for frame in frames:

        if frame.camera_pose is None:
            continue

        T_wc = pose_to_matrix(frame.camera_pose)

        for state in frame.object_states:

            if state.object_id not in trajectory_data:

                print("=" * 80)
                print("ERROR")
                print("Frame:", frame.frame_id)
                print("Timestamp:", frame.timestamp)
                print("Object:", state.object_id)
                print("Existing keys:", sorted(trajectory_data.keys()))
                print("=" * 80)

                raise KeyError(state.object_id)

            T_co = pose_to_matrix(state.pose)
            T_wo = T_wc @ T_co

            trajectory_data[state.object_id]["timestamps"].append(
                frame.timestamp
            )

            trajectory_data[state.object_id]["positions"].append(
                T_wo[:3, 3]
            )

    trajectories = {}

    for object_id, values in trajectory_data.items():

        if len(values["timestamps"]) == 0:
            continue

        trajectories[object_id] = ObjectTrajectory(
            object_id=object_id,
            timestamps=np.asarray(values["timestamps"]),
            positions=np.asarray(values["positions"]),
        )

    return trajectories

In [49]:
traj = debug_collect_world_trajectories(
    dataset.frames,
    dataset.objects,
)

print("Success!")
print("Trajectories:", len(traj))

Initial keys:
[5, 6, 8, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26, 27, 28]

Success!
Trajectories: 18


In [50]:
from pathlib import Path
import numpy as np
import json

ROOT = Path("/home/ayon/HDD/EventDatasets/EVIMO2_official")

seqs = [
    ROOT / "flea3_7/sfm/eval/scene_03_02_000002",
    ROOT / "flea3_7/sfm/train/seq_1_0_000003",
]

seqs

[PosixPath('/home/ayon/HDD/EventDatasets/EVIMO2_official/flea3_7/sfm/eval/scene_03_02_000002'),
 PosixPath('/home/ayon/HDD/EventDatasets/EVIMO2_official/flea3_7/sfm/train/seq_1_0_000003')]

In [51]:
for seq in seqs:
    print("="*80)
    print(seq)

    info = np.load(seq/"dataset_info.npz", allow_pickle=True)

    print(info.files)

    for k in info.files:
        arr = info[k]
        print(
            f"{k:30s}",
            type(arr),
            getattr(arr, "shape", None)
        )

/home/ayon/HDD/EventDatasets/EVIMO2_official/flea3_7/sfm/eval/scene_03_02_000002
['index', 'discretization', 'K', 'D', 'meta']
index                          <class 'numpy.ndarray'> (0,)
discretization                 <class 'numpy.ndarray'> ()
K                              <class 'numpy.ndarray'> (3, 3)
D                              <class 'numpy.ndarray'> (4,)
meta                           <class 'numpy.ndarray'> ()
/home/ayon/HDD/EventDatasets/EVIMO2_official/flea3_7/sfm/train/seq_1_0_000003
['index', 'discretization', 'K', 'D', 'meta']
index                          <class 'numpy.ndarray'> (0,)
discretization                 <class 'numpy.ndarray'> ()
K                              <class 'numpy.ndarray'> (3, 3)
D                              <class 'numpy.ndarray'> (4,)
meta                           <class 'numpy.ndarray'> ()


In [52]:
for seq in seqs:

    print("\n")
    print("="*80)
    print(seq.name)

    info = np.load(
        seq/"dataset_info.npz",
        allow_pickle=True
    )

    for k in info.files:

        print("\nKEY:", k)

        x = info[k]

        if isinstance(x, np.ndarray):
            print(x[:2])

        else:
            print(x)



scene_03_02_000002

KEY: index
[]

KEY: discretization


IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [ ]:
from pathlib import Path
import numpy as np
import json

root = Path("/home/ayon/HDD/EventDatasets/EVIMO2_official")

seqs = [
    root / "flea3_7/sfm/eval/scene_03_02_000002",
    root / "flea3_7/sfm/train/seq_1_0_000003"
]

for seq in seqs:
    print("="*80)
    print(seq)
    
    info = np.load(seq / "dataset_info.npz", allow_pickle=True)

    print(info.files)

    for k in info.files:
        x = info[k]
        print(k, type(x), x.shape if hasattr(x,"shape") else "")

In [ ]:
for seq in seqs:

    print("\n", "="*80)
    print(seq.name)

    info = np.load(seq / "dataset_info.npz", allow_pickle=True)

    meta = info["meta"]

    print("meta type:", type(meta))
    print("meta shape:", meta.shape)

    # unwrap numpy object if needed
    try:
        meta = meta.item()
    except:
        pass

    print("meta type after unwrap:", type(meta))

    if isinstance(meta, dict):
        print("meta keys:")
        for k in meta.keys():
            print("  ", k)
    else:
        print(meta)

In [ ]:
for seq in seqs:

    print("\n", "="*80)
    print(seq.name)

    masks = np.load(seq/"dataset_mask.npz", allow_pickle=True)

    print("Mask arrays:", masks.files)

    for k in masks.files:
        arr = masks[k]

        ids = np.unique(arr)

        if 12 in ids:
            print("\n", k)
            print("shape:", arr.shape)
            print("dtype:", arr.dtype)
            print("number of IDs:", len(ids))
            print("first IDs:", ids[:30])
            print("contains ID 12:", 12 in ids)

In [ ]:
for seq in seqs:

    print("\n", "="*80)
    print(seq.name)

    masks = np.load(seq/"dataset_mask.npz", allow_pickle=True)

    for k in masks.files:

        arr = masks[k]

        ids = np.unique(arr)

        print(k)
        print("IDs:", ids[:50])
        print("Max ID:", ids.max())

In [ ]:
seq = seqs[1]

masks = np.load(seq/"dataset_mask.npz", allow_pickle=True)

for k in masks.files:

    arr = masks[k]

    if 12 in np.unique(arr):

        print("Found object 12 in:", k)

        frames = np.where(
            np.any(arr == 12, axis=(1,2))
        )[0]

        print("Frames:", frames[:20])
        print("Count:", len(frames))

In [ ]:
for seq in seqs:

    print("\n", "="*80)
    print(seq.name)

    info = np.load(seq / "dataset_info.npz", allow_pickle=True)
    meta = info["meta"].item()

    print("full_trajectory type:", type(meta["full_trajectory"]))

    ft = meta["full_trajectory"]

    if isinstance(ft, dict):
        print("trajectory object IDs:")
        print(list(ft.keys())[:50])
        print("contains 12:", 12 in ft)

    elif isinstance(ft, np.ndarray):
        print("shape:", ft.shape)
        print("dtype:", ft.dtype)
        print("first entries:")
        print(ft[:3])

In [ ]:
for seq in seqs:

    print("\n", "="*80)
    print(seq.name)

    info = np.load(seq / "dataset_info.npz", allow_pickle=True)
    meta = info["meta"].item()

    ft = meta["full_trajectory"]

    print("number of trajectories:", len(ft))

    for i, item in enumerate(ft[:5]):
        print("\nTrajectory", i)
        print("type:", type(item))

        if isinstance(item, dict):
            print("keys:", item.keys())

        elif isinstance(item, np.ndarray):
            print("shape:", item.shape)
            print("dtype:", item.dtype)

        else:
            print(str(item)[:200])

In [ ]:
for seq in seqs:

    print("\n", "="*80)
    print(seq.name)

    info = np.load(seq / "dataset_info.npz", allow_pickle=True)
    meta = info["meta"].item()

    ft = meta["full_trajectory"]

    ids = []

    for item in ft:
        if isinstance(item, dict):
            for key in ["id", "object_id", "obj_id", "instance_id"]:
                if key in item:
                    ids.append(item[key])

    print("Extracted IDs:", ids[:50])
    print("Contains 12:", 12 in ids)

In [ ]:
for seq in seqs:

    print("\n", "="*80)
    print(seq.name)

    meta = np.load(
        seq/"dataset_info.npz",
        allow_pickle=True
    )["meta"].item()

    frame0 = meta["full_trajectory"][0]

    print("integer 12:", 12 in frame0)
    print("string 12 :", "12" in frame0)


In [ ]:
from src.data.evimo2.parser import EVIMO2Parser
from pathlib import Path

seq = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/flea3_7/sfm/eval/scene_03_02_000002"
)

dataset = EVIMO2Parser(seq)

print("Object dictionary keys:")
print(list(dataset.objects.keys())[:20])

print("\nKey types:")
print([type(k) for k in list(dataset.objects.keys())[:10]])

print("\nFrame object state IDs:")
ids = []

for frame in dataset.frames:
    for state in frame.object_states:
        ids.append(state.object_id)

print(ids[:20])
print([type(x) for x in ids[:20]])

In [ ]:
from pathlib import Path
from src.data.evimo2.parser import EVIMO2Parser

seq = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/flea3_7/sfm/eval/scene_03_02_000002"
)

dataset = EVIMO2Parser(seq)

print("Number of objects:", len(dataset.objects))

print("Object IDs:")
print(sorted(dataset.objects.keys()))

print("\nFrame object IDs:")

ids = set()

for frame in dataset.frames:
    for state in frame.object_states:
        ids.add(state.object_id)

print(sorted(ids))

print("\nMissing:")
print(ids - set(dataset.objects.keys()))

In [ ]:
from src.preprocessing.object_motion import collect_world_trajectories

trajectories = collect_world_trajectories(
    dataset.frames,
    dataset.objects,
)

print("Trajectories:", len(trajectories))

print("IDs:")
print(sorted(trajectories.keys()))

In [ ]:
from pathlib import Path
from src.data.evimo2.parser import EVIMO2Parser
from src.preprocessing.frame_motion import load_frame_motion


root = Path("/home/ayon/HDD/EventDatasets/EVIMO2_official")

seqs = [
    root / "flea3_7/sfm/eval/scene_03_02_000002",
    root / "flea3_7/sfm/train/seq_1_0_000003",
]


for seq in seqs:

    print("\n" + "="*80)
    print(seq.name)

    dataset = EVIMO2Parser(seq)

    cache = load_frame_motion(seq)

    print("Frames:")
    print(" parser:", len(dataset.frames))
    print(" cache :", len(cache.frame_ids))

    print("Objects:")
    print(" parser:", len(dataset.objects))
    print(" cache :", len(cache.object_ids))

    print("Object IDs:")
    print(cache.object_ids)

    print("Motion shape:")
    print(cache.delta_position.shape)


In [ ]:
from pathlib import Path
from src.preprocessing.frame_motion import load_frame_motion
import numpy as np
for seq in seqs:

    print("\n" + "="*80)
    print(seq.name)

    cache = load_frame_motion(seq)

    speed = cache.speed

    print("Speed statistics")
    print("----------------")

    print("Minimum :", speed.min())
    print("Mean    :", speed.mean())
    print("Median  :", np.median(speed))
    print("Maximum :", speed.max())


    print("\nPer-object maximum speed")

    for obj, col in zip(
        cache.object_ids,
        range(len(cache.object_ids))
    ):
        print(
            f"Object {obj:2d}: "
            f"{speed[:,col].max():.4f} m/s"
        )

In [ ]:
for seq in seqs:

    print("\n", seq.name)

    cache = load_frame_motion(seq)

    displacement = np.linalg.norm(
        cache.delta_position,
        axis=2,
    )

    print(
        "Maximum displacement:",
        displacement.max(),
        "m"
    )

    print(
        "Mean displacement:",
        displacement.mean(),
        "m"
    )

In [ ]:
import matplotlib.pyplot as plt

for seq in seqs:

    cache = load_frame_motion(seq)

    print("\n", seq.name)

    for obj_id in cache.object_ids:

        col = list(cache.object_ids).index(obj_id)

        max_speed = cache.speed[:, col].max()

        if max_speed > 0.05:

            plt.figure(figsize=(8,3))
            plt.plot(
                cache.timestamps,
                cache.speed[:, col]
            )

            plt.title(
                f"{seq.name} - Object {obj_id}"
            )

            plt.xlabel("time (s)")
            plt.ylabel("speed (m/s)")
            plt.grid()

            plt.show()

In [ ]:
from pathlib import Path
import numpy as np

from src.preprocessing.frame_motion import load_frame_motion


root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

motion_files = list(
    root.glob("**/cache/frame_motion.npz")
)

print("Found caches:", len(motion_files))


all_speeds = []

for path in motion_files:

    seq = path.parent.parent

    cache = load_frame_motion(seq)

    speeds = cache.speed.flatten()

    # remove the artificial first-frame zeros
    speeds = speeds[speeds > 0]

    all_speeds.append(speeds)


all_speeds = np.concatenate(all_speeds)


print("Total motion samples:", len(all_speeds))

In [ ]:
print("Minimum :", all_speeds.min())
print("Maximum :", all_speeds.max())

print("\nPercentiles")

for p in [
    50,
    75,
    90,
    95,
    97,
    99,
    99.5,
    99.9,
]:

    print(
        f"{p:5.1f}% : {np.percentile(all_speeds,p):.6f} m/s"
    )

In [ ]:
thresholds = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.1,
]


print(
    f"{'Threshold':>12} {'Dynamic %':>12}"
)

print("-"*30)


for t in thresholds:

    ratio = (
        np.sum(all_speeds > t)
        /
        len(all_speeds)
        *
        100
    )

    print(
        f"{t:12.4f} {ratio:12.3f}"
    )

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(8,4))

plt.hist(
    all_speeds,
    bins=200,
)

plt.yscale("log")

plt.xlabel("Object speed (m/s)")
plt.ylabel("Count")

plt.title(
    "EVIMO2 object motion distribution"
)

plt.grid()

plt.show()

In [ ]:
for path in motion_files[:10]:

    seq = path.parent.parent

    cache = load_frame_motion(seq)

    speeds = cache.speed.flatten()
    speeds = speeds[speeds > 0]

    print(
        seq.name,
        "median:",
        np.median(speeds),
        "max:",
        np.max(speeds)
    )

In [ ]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

# -
from src.preprocessing.event_index import compute_event_index
from src.data.evimo2.parser import EVIMO2Parser



from pathlib import Path

from src.data.dataset import EVIMO2Dataset

dataset = EVIMO2Dataset(
    dataset_root=Path("/home/ayon/HDD/EventDatasets/EVIMO2_official"),
    split="train",
)

In [ ]:
sample = dataset[0]

In [ ]:
print("Sequence :", sample.sequence_name)
print("Sensor   :", sample.sensor)
print("Frame ID :", sample.frame_id)
print("Timestamp:", sample.timestamp)

In [ ]:
print(type(sample.rgb))

if sample.rgb is None:
    print("RGB not available")
else:
    print(sample.rgb.shape)
    print(sample.rgb.dtype)

In [ ]:
import matplotlib.pyplot as plt


fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].imshow(sample.rgb)
ax[0].set_title("RGB")
ax[0].axis("off")

ax[1].imshow(sample.depth)
ax[1].set_title("Depth")
ax[1].axis("off")

plt.show()

In [ ]:
indices = [0, 10, 50, 100, 500]

for idx in indices:
    s = dataset[idx]

    print(
        idx,
        s.sequence_name,
        s.frame_id,
        s.rgb is not None,
        None if s.rgb is None else s.rgb.shape,
    )

In [ ]:
reader = dataset.readers[0]

print("Dataset frame:", sample.frame_id)

print("RGB exists:",
      f"classical_{sample.frame_id:010d}" in reader._rgb.files)

In [ ]:
<class 'src.data.sample.EVIMO2Sample'>
EVIMO2Sample(sequence_name='scene13_dyn_test_01_000000', sensor='left_camera', local_frame_index=0, frame_id=0, timestamp=0.016667, events_xy=memmap([[463, 425],
        [578, 425],
        [634, 427],
        ...,
        [225,   3],
        [ 96,   7],
        [ 25,   6]], shape=(98954, 2), dtype=uint16), events_t=memmap([0.01666778, 0.01666778, 0.01666778, ..., 0.03333261, 0.03333261,
        0.03333261], shape=(98954,), dtype=float32), events_p=memmap([1, 0, 1, ..., 0, 0, 0], shape=(98954,), dtype=uint8), camera_motion=CameraMotion(translation=array([ 0.002478,  0.000536, -0.000898], dtype=float32), quaternion=array([-2.46000e-03,  9.81000e-04, -4.07200e-03,  9.99988e-01],
      dtype=float32), delta_translation=array([0., 0., 0.], dtype=float32), delta_quaternion=array([0., 0., 0., 1.], dtype=float32), linear_speed=0.0, angular_speed=0.0, dt=0.016666000708937645, pose_available=True), frame_motion=FrameMotion(object_ids=array([ 5,  6, 11, 14, 15, 22, 23, 24], dtype=int32), delta_position=array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]], dtype=float32), speed=array([0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)), depth=array([[1559, 1554, 1551, ...,  590,  590,  589],
       [1560, 1555, 1553, ...,  590,  590,  589],
       [1560, 1557, 1553, ...,  590,  590,  589],
       ...,
       [   0,    0,    0, ...,    0,    0,    0],
       [   0,    0,    0, ...,    0,    0,    0],
       [   0,    0,    0, ...,    0,    0,    0]],
      shape=(480, 640), dtype=uint16), mask=array([[22000, 22000, 22000, ..., 22000, 22000, 22000],
       [22000, 22000, 22000, ..., 22000, 22000, 22000],
       [22000, 22000, 22000, ..., 22000, 22000, 22000],
       ...,
       [    0,     0,     0, ...,     0,     0,     0],
       [    0,     0,     0, ...,     0,     0,     0],
       [    0,     0,     0, ...,     0,     0,     0]],
      shape=(480, 640), dtype=uint16), rgb=None)

In [ ]:
reader = dataset.readers[0]

print(reader._rgb.files[:10])

In [ ]:
sample = dataset[0]

key = f"classical_{sample.frame_id:010d}"

print(key)
print(key in reader._rgb.files)

In [ ]:
from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset

frame_dataset = EVIMO2Dataset(
    dataset_root="~/HDD/EventDatasets/EVIMO2_official",
    split="train",
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)

print(len(temporal_dataset))

In [ ]:
sample = temporal_dataset[0]

print(sample.global_indices)
print(sample.history_offsets)
print(len(sample.frames))

In [ ]:
for frame in sample.frames:
    print(frame.frame_id, frame.timestamp)

In [ ]:
for frame in sample.frames:
    print(frame.sequence_name)

In [ ]:
temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-6, -4, -2, 0),
)

sample = temporal_dataset[5]

for frame in sample.frames:
    print(frame.frame_id)

In [ ]:
sample = temporal_dataset[115]

for frame in sample.frames:
    print(
        "local =", frame.local_frame_index,
        "frame_id =", frame.frame_id,
        "timestamp =", frame.timestamp,
    )

In [ ]:
from tqdm import tqdm

bad_samples = []

for sample_idx in tqdm(range(len(temporal_dataset))):

    sample = temporal_dataset[sample_idx]

    sequence_names = [f.sequence_name for f in sample.frames]

    if len(set(sequence_names)) != 1:

        bad_samples.append(
            (
                sample_idx,
                sequence_names,
                [f.local_frame_index for f in sample.frames],
                [f.frame_id for f in sample.frames],
            )
        )

print("=" * 80)

if len(bad_samples) == 0:
    print("PASS")
    print("All temporal samples belong to a single sequence.")
else:
    print(f"FAILED : {len(bad_samples)} invalid samples found.\n")

    for sample_idx, seqs, local_ids, frame_ids in bad_samples[:10]:

        print(f"Temporal sample {sample_idx}")
        print("Sequences   :", seqs)
        print("Local index :", local_ids)
        print("Frame IDs   :", frame_ids)
        print("-" * 80)

In [ ]:
bad_sensor = []

for sample_idx in tqdm(range(len(temporal_dataset))):

    sample = temporal_dataset[sample_idx]

    sensors = [f.sensor for f in sample.frames]

    if len(set(sensors)) != 1:

        bad_sensor.append(
            (
                sample_idx,
                sensors,
            )
        )

print("=" * 80)

if len(bad_sensor) == 0:
    print("PASS")
    print("All temporal samples belong to one sensor.")
else:
    print(f"FAILED : {len(bad_sensor)} invalid samples found.")

In [56]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)


from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

frame_dataset = EVIMO2Dataset(
    dataset_root="~/HDD/EventDatasets/EVIMO2_official",
    split="train",
    sensors="left_camera"
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3,-2,-1,0),
)

len(frame_dataset), len(temporal_dataset)

Project Root: /home/ayon/git/EventCameraProject
EVIMO2 Sequence Index
Sequences : 0
Frames    : 0
Sensors   : l, e, f, t, _, c, a, m, e, r, a
Split     : train


(0, 0)

In [57]:
sample = frame_dataset[0]

print("="*80)
print("CAMERA CALIBRATION TEST")
print("="*80)

print()

print("K:")
print(sample.camera_intrinsics)

print()

print("D:")
print(sample.camera_distortion)

print()

print("Shapes")
print(
    "K:",
    sample.camera_intrinsics.shape
)

print(
    "D:",
    sample.camera_distortion.shape
)

IndexError: list index out of range

In [ ]:
loader = DataLoader(
    temporal_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=temporal_collate_fn,
)

batch = next(iter(loader))

type(batch)

In [ ]:
len(batch.frames)

In [ ]:
batch.history_offsets

In [ ]:
import numpy as np
for t, frame_batch in enumerate(batch.frames):

    print("=" * 70)
    print(f"Timestep {t}")
    print(f"Offset {batch.history_offsets[t]}")
    print()

    print(type(frame_batch))
    print()

    print("Sequence names :", frame_batch.sequence_names)
    print("Frame IDs      :", frame_batch.frame_ids)
    print("Local indices  :", frame_batch.local_frame_indices)
    print("Timestamps     :", frame_batch.timestamps)

    print()

    print("Events XY :", frame_batch.events_xy.shape)
    print("Events T  :", frame_batch.events_t.shape)
    print("Events P  :", frame_batch.events_p.shape)

    print("Event ownership :", np.unique(frame_batch.event_sample_indices))

    print()

    if frame_batch.depth is not None:
        print("Depth :", frame_batch.depth.shape)

    if frame_batch.mask is not None:
        print("Mask  :", frame_batch.mask.shape)

    if frame_batch.rgb is not None:
        print("RGB   :", frame_batch.rgb.shape)

In [ ]:
for sample in range(2):

    print(f"Sample {sample}")

    for frame_batch in batch.frames:

        print(frame_batch.frame_ids[sample], end=" ")

    print()

In [ ]:
frame_batch = batch.frames[-1]

for sample in range(2):

    count = np.sum(
        frame_batch.event_sample_indices == sample
    )

    print(sample, count)

In [ ]:
frame_batch = batch.frames[-1]

for sample in range(2):

    print("Sample", sample)
    print("Frame", frame_batch.frame_ids[sample])
    print("Objects", frame_batch.frame_motion[sample].object_ids)
    print("Speeds", frame_batch.frame_motion[sample].speed)
    print()

In [ ]:
for t, frame_batch in enumerate(batch.frames):

    print(f"\n===== Offset {batch.history_offsets[t]} =====")

    for sample_idx, motion in enumerate(frame_batch.camera_motion):

        print(
            sample_idx,
            motion.translation,
            motion.linear_speed,
            motion.angular_speed,
        )

In [ ]:
import numpy as np

print("="*100)
print("TEMPORAL COLLATE VERIFICATION")
print("="*100)

print(f"Temporal length : {len(batch.frames)}")
print(f"History offsets : {batch.history_offsets}")

batch_size = len(batch.frames[0].frame_ids)

print(f"Batch size      : {batch_size}")

print()

################################################################################
# Verify every timestep
################################################################################

for t, frame_batch in enumerate(batch.frames):

    print("="*100)
    print(f"Timestep {t}   Offset {batch.history_offsets[t]}")
    print("="*100)

    #
    # Metadata
    #

    print("Sequence names")
    print(frame_batch.sequence_names)

    print()

    print("Sensors")
    print(frame_batch.sensors)

    print()

    print("Frame IDs")
    print(frame_batch.frame_ids)

    print()

    print("Local frame indices")
    print(frame_batch.local_frame_indices)

    print()

    print("Timestamps")
    print(frame_batch.timestamps)

    print()

    #
    # Events
    #

    print("Events")

    print("events_xy :", frame_batch.events_xy.shape)
    print("events_t  :", frame_batch.events_t.shape)
    print("events_p  :", frame_batch.events_p.shape)

    print()

    #
    # Event ownership
    #

    unique, counts = np.unique(
        frame_batch.event_sample_indices,
        return_counts=True,
    )

    print("Event ownership")

    for u, c in zip(unique, counts):

        print(f"sample {u} -> {c:,} events")

    print()

    #
    # Dense tensors
    #

    if frame_batch.depth is not None:

        print("Depth :", frame_batch.depth.shape)

    if frame_batch.mask is not None:

        print("Mask  :", frame_batch.mask.shape)

    if frame_batch.rgb is not None:

        print("RGB   :", frame_batch.rgb.shape)

    print()

    #
    # Motion
    #

    print("Camera motions :", len(frame_batch.camera_motion))
    print("Frame motions  :", len(frame_batch.frame_motion))

    print()

In [ ]:
print("="*100)
print("VERIFY TEMPORAL ORDER")
print("="*100)

for sample_idx in range(batch_size):

    print()

    print("-"*80)
    print(f"Sample {sample_idx}")
    print("-"*80)

    for offset, frame_batch in zip(

        batch.history_offsets,

        batch.frames,

    ):

        motion = frame_batch.camera_motion[sample_idx]

        fmotion = frame_batch.frame_motion[sample_idx]

        event_count = np.sum(
            frame_batch.event_sample_indices == sample_idx
        )

        print(

            f"offset={offset:>3} | "

            f"sensor={frame_batch.sensors[sample_idx]:15s} | "

            f"sequence={frame_batch.sequence_names[sample_idx]} | "

            f"frame={frame_batch.frame_ids[sample_idx]:4d} | "

            f"local={frame_batch.local_frame_indices[sample_idx]:4d} | "

            f"time={frame_batch.timestamps[sample_idx]:.6f} | "

            f"events={event_count:7d} | "

            f"objects={len(fmotion.object_ids):2d} | "

            f"camera_speed={motion.linear_speed:.5f}"

        )

In [ ]:
import matplotlib.pyplot as plt

sample_idx = 0

fig, ax = plt.subplots(
    len(batch.frames),
    3,
    figsize=(12,16),
)

for row, frame_batch in enumerate(batch.frames):

    ax[row,0].imshow(
        frame_batch.rgb[sample_idx]
    )

    ax[row,0].set_title(
        f"RGB {batch.history_offsets[row]}"
    )

    ax[row,1].imshow(
        frame_batch.depth[sample_idx]
    )

    ax[row,1].set_title(
        "Depth"
    )

    ax[row,2].imshow(
        frame_batch.mask[sample_idx]
    )

    ax[row,2].set_title(
        "Mask"
    )

    for c in range(3):
        ax[row,c].axis("off")

plt.tight_layout()

In [ ]:
print("="*100)
print("CAMERA MOTION")
print("="*100)

sample_idx = 0

for offset, frame_batch in zip(

    batch.history_offsets,

    batch.frames,

):

    m = frame_batch.camera_motion[sample_idx]

    print(

        offset,

        "\ntranslation      :", m.translation,

        "\ndelta_translation:", m.delta_translation,

        "\nlinear_speed     :", m.linear_speed,

        "\nangular_speed    :", m.angular_speed,

        "\n"

    )

In [ ]:
print("="*100)
print("OBJECT MOTION")
print("="*100)

sample_idx = 0

for offset, frame_batch in zip(

    batch.history_offsets,

    batch.frames,

):

    m = frame_batch.frame_motion[sample_idx]

    print()

    print(f"Offset {offset}")

    print("Object IDs")

    print(m.object_ids)

    print()

    print("Speeds")

    print(m.speed)

    print()

    print("Delta position shape")

    print(m.delta_position.shape)

In [ ]:
print("="*100)
print("VERIFY SAME SEQUENCE")
print("="*100)

for sample_idx in range(batch_size):

    sequences = [

        frame.sequence_names[sample_idx]

        for frame in batch.frames

    ]

    sensors = [

        frame.sensors[sample_idx]

        for frame in batch.frames

    ]

    print()

    print("Sample", sample_idx)

    print("Sequences :", sequences)

    print("Sensors   :", sensors)

    assert len(set(sequences)) == 1
    assert len(set(sensors)) == 1

print()

print("PASS")

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    temporal_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

batch = next(iter(loader))

In [ ]:
frame = batch.frames[0]

print(type(frame.depth))
print(type(frame.mask))
print(type(frame.rgb))

print(len(frame.depth))
print(len(frame.mask))
print(len(frame.rgb))

for i in range(len(frame.rgb)):
    print(
        i,
        frame.sensors[i],
        None if frame.rgb[i] is None else frame.rgb[i].shape,
        None if frame.depth[i] is None else frame.depth[i].shape,
        None if frame.mask[i] is None else frame.mask[i].shape,
    )

In [ ]:
from pathlib import Path
import numpy as np

sequence = Path(
    "~/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000"
).expanduser()

print("Files:")
for f in sorted(sequence.iterdir()):
    print(" ", f.name)

In [ ]:
import numpy as np
from pathlib import Path

info = np.load(
    Path(
        "~/HDD/EventDatasets/EVIMO2_official/samsung_mono/imo/train/scene13_dyn_test_01_000000/dataset_info.npz"
    ).expanduser(),
    allow_pickle=True,
)

print(info.files)



for key in info.files:

    arr = info[key]

    print("="*80)
    print(key)
    print(type(arr))
    print(arr.dtype)
    print(arr.shape)

    if arr.ndim > 0:
        print(arr[:3])

In [ ]:
from pprint import pprint

meta = info["meta"].item()

print(type(meta))

print(meta.keys())

imu = meta["imu"]

print(type(imu))

if isinstance(imu, dict):
    print(imu.keys())
else:
    print(len(imu))
    print(type(imu[0]))


from pprint import pprint

if isinstance(imu, dict):
    for k, v in imu.items():
        print("=" * 80)
        print(k)
        print(type(v))
        if isinstance(v, np.ndarray):
            print(v.shape, v.dtype)
            print(v[:5])
        else:
            pprint(v)
else:
    pprint(imu[:3])

In [ ]:
from pprint import pprint
import numpy as np

imu = meta["imu"]

print(type(imu))

if isinstance(imu, dict):

    for key, value in imu.items():

        print("\n", "=" * 80)
        print(key)

        print(type(value))

        if isinstance(value, np.ndarray):
            print("shape :", value.shape)
            print("dtype :", value.dtype)
            print(value[:3])

        else:
            pprint(value)

In [ ]:
print(imu['/prophesee/left/imu'].dtype)

In [ ]:
print(type(imu))
print(len(imu))

print(imu['/prophesee/left/imu'][:1])
print(imu['/prophesee/right/imu'][:1])


In [ ]:
!ls ~/HDD/EventDatasets/EVIMO2_official/samsung_mono/imo/train/

In [ ]:
from pathlib import Path
import numpy as np

sequence = Path(
    "~/HDD/EventDatasets/EVIMO2_official/samsung_mono/imo/train/scene10_dyn_train_00_000000"
).expanduser()

info = np.load(sequence / "dataset_info.npz", allow_pickle=True)

meta = info["meta"].item()

print(meta.keys())
print()
print("IMU type:", type(meta["imu"]))
print("IMU keys:")
print(meta["imu"].keys())
print()

for key, value in meta["imu"].items():
    print("=" * 80)
    print(key)
    print(type(value))
    print("Length:", len(value))
    print("First sample:")
    print(value[0])

In [ ]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)


In [ ]:
from src.data.evimo2._metadata import parse_sequence_metadata

sequence_dir = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000/"
)

frames, objects, imu = parse_sequence_metadata(sequence_dir)

print(imu.keys())

for sensor, data in imu.items():

    print("=" * 80)
    print(sensor)

    print("timestamps :", data["timestamps"].shape)
    print("gyro       :", data["gyro"].shape)
    print("acc        :", data["acceleration"].shape)

    print(data["timestamps"][:5])
    print(data["gyro"][:2])
    print(data["acceleration"][:2])

In [ ]:
from pathlib import Path
import numpy as np
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

# ============================================================
# Dataset
# ============================================================

dataset_root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

frame_dataset = EVIMO2Dataset(
    dataset_root=dataset_root,
    sensors=(
        "left_camera",
        "right_camera",
    ),                 # all sensors
    split="train",
    load_depth=True,
    load_mask=True,
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)

loader = DataLoader(
    temporal_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

batch = next(iter(loader))

print("=" * 100)
print("TEMPORAL BATCH VERIFICATION")
print("=" * 100)

print("Temporal length :", len(batch.frames))
print("History offsets :", batch.history_offsets)
print()

# ============================================================
# Verify every timestep
# ============================================================

for timestep, frame_batch in enumerate(batch.frames):

    print("=" * 100)
    print(
        f"Timestep {timestep}   Offset {batch.history_offsets[timestep]}"
    )
    print("=" * 100)

    print()

    print("Sequences")
    print(frame_batch.sequence_names)

    print()

    print("Sensors")
    print(frame_batch.sensors)

    print()

    print("Frame IDs")
    print(frame_batch.frame_ids)

    print()

    print("Local frame indices")
    print(frame_batch.local_frame_indices)

    print()

    print("Timestamps")
    print(frame_batch.timestamps)

    print()

    print("Events")
    print("XY :", frame_batch.events_xy.shape)
    print("T  :", frame_batch.events_t.shape)
    print("P  :", frame_batch.events_p.shape)

    print()

    print("Event ownership")

    for sample_id in range(len(frame_batch.sequence_names)):

        n = np.sum(
            frame_batch.event_sample_indices == sample_id
        )

        print(
            f"sample {sample_id}: {n:,} events"
        )

    print()

    print("Camera motions :", len(frame_batch.camera_motion))
    print("Frame motions  :", len(frame_batch.frame_motion))
    print("IMU windows    :", len(frame_batch.imu))

    print()

    print("Depth entries :", len(frame_batch.depth))
    print("Mask entries  :", len(frame_batch.mask))
    print("RGB entries   :", len(frame_batch.rgb))

print()

# ============================================================
# Verify temporal consistency
# ============================================================

print("=" * 100)
print("VERIFY TEMPORAL CONSISTENCY")
print("=" * 100)

batch_size = len(batch.frames[0].sequence_names)

for sample in range(batch_size):

    print()
    print("-" * 100)
    print(f"Sample {sample}")
    print("-" * 100)

    sequence_names = []
    sensors = []

    previous_timestamp = None

    for timestep, frame_batch in enumerate(batch.frames):

        sequence = frame_batch.sequence_names[sample]
        sensor = frame_batch.sensors[sample]

        sequence_names.append(sequence)
        sensors.append(sensor)

        frame_id = frame_batch.frame_ids[sample]
        local = frame_batch.local_frame_indices[sample]
        timestamp = frame_batch.timestamps[sample]

        cam = frame_batch.camera_motion[sample]
        motion = frame_batch.frame_motion[sample]
        imu = frame_batch.imu[sample]

        n_events = np.sum(
            frame_batch.event_sample_indices == sample
        )

        print(
            f"offset={batch.history_offsets[timestep]:3d} | "
            f"{sensor:15s} | "
            f"{sequence:35s} | "
            f"frame={frame_id:4d} | "
            f"time={timestamp:.6f} | "
            f"events={n_events:7d} | "
            f"objects={len(motion.object_ids):2d} | "
            f"imu={len(imu.timestamps):4d}"
        )

        #
        # timestamps should increase
        #

        if previous_timestamp is not None:

            assert timestamp > previous_timestamp

        previous_timestamp = timestamp

    #
    # all sequence names identical
    #

    assert len(set(sequence_names)) == 1

    #
    # all sensors identical
    #

    assert len(set(sensors)) == 1

print()

print("=" * 100)
print("VERIFY IMU")
print("=" * 100)

for timestep, frame_batch in enumerate(batch.frames):

    print()
    print(f"Offset {batch.history_offsets[timestep]}")

    imu = frame_batch.imu[0]

    print("IMU samples :", len(imu.timestamps))

    if len(imu.timestamps):

        print("first :", imu.timestamps[0])
        print("last  :", imu.timestamps[-1])

        print("gyro shape :", imu.angular_velocity.shape)
        print("acc  shape :", imu.linear_acceleration.shape)

        assert (
            imu.angular_velocity.shape[0]
            ==
            len(imu.timestamps)
        )

        assert (
            imu.linear_acceleration.shape[0]
            ==
            len(imu.timestamps)
        )

print()

print("=" * 100)
print("VERIFY OPTIONAL DATA")
print("=" * 100)

for timestep, frame_batch in enumerate(batch.frames):

    print()
    print(f"Offset {batch.history_offsets[timestep]}")

    for i in range(batch_size):

        sensor = frame_batch.sensors[i]

        rgb = frame_batch.rgb[i]
        depth = frame_batch.depth[i]
        mask = frame_batch.mask[i]

        print(
            i,
            sensor,
            None if rgb is None else rgb.shape,
            None if depth is None else depth.shape,
            None if mask is None else mask.shape,
        )

print()
print("=" * 100)
print("ALL CHECKS PASSED")
print("=" * 100)

In [ ]:
from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
)

pipeline = Compose([
    ToTensor(),
    NormalizeEventTime(),
    NormalizeIMU(),
])

from pathlib import Path
import numpy as np
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

# ============================================================
# Dataset
# ============================================================

dataset_root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

frame_dataset = EVIMO2Dataset(
    dataset_root=dataset_root,
    sensors=(
        "left_camera",
        "right_camera",
    ),                 # all sensors
    split="train",
    load_depth=True,
    load_mask=True,
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)

loader = DataLoader(
    temporal_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

batch = next(iter(loader))


tensor_batch = pipeline(batch)

print(type(tensor_batch))

frame = tensor_batch.frames[0]

print(frame.events_xy.dtype)
print(frame.events_t.dtype)
print(frame.events_p.dtype)

print(frame.events_t.min(), frame.events_t.max())

imu = frame.imu[0]

print(imu.timestamps.min(), imu.timestamps.max())
print(imu.angular_velocity.dtype)
print(imu.linear_acceleration.dtype)

In [ ]:
import torch
tensor_batch = pipeline(batch)

frame = tensor_batch.frames[0]

for sample_id in torch.unique(frame.event_sample_indices):

    mask = frame.event_sample_indices == sample_id

    t = frame.events_t[mask]

    print(
        sample_id.item(),
        t.min().item(),
        t.max().item(),
        len(t),
    )

In [ ]:
frame = tensor_batch.frames[0]

print("Number of events:", frame.events_t.numel())

print("First 10 timestamps:")
print(frame.events_t[:10])

print("Min:", frame.events_t.min())
print("Max:", frame.events_t.max())

In [ ]:
frame = batch.frames[0]

print(type(frame.events_t))
print(frame.events_t.dtype)
print(frame.events_t.shape)

print(frame.events_t[:10])

print(frame.events_t.min())
print(frame.events_t.max())

In [ ]:
import numpy as np

t = np.load(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000/dataset_events_t.npy",
    mmap_mode="r",
)

print(t.dtype)
print(t.shape)

print(t[:10])

print(t.min())
print(t.max())

In [ ]:
from src.data.evimo2._reader import EVIMO2Reader

reader = EVIMO2Reader(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official/left_camera/imo/train/scene13_dyn_test_01_000000"
)

print(reader.events_t.dtype)
print(reader.events_t.shape)

print(reader.events_t[:10])

print(reader.events_t.min())
print(reader.events_t.max())

In [ ]:
sample = frame_dataset[0]

print(sample.events_t.dtype)
print(sample.events_t.shape)

print(sample.events_t[:10])

print(sample.events_t.min())
print(sample.events_t.max())

In [ ]:
temporal_sample = temporal_dataset[0]

for i, frame in enumerate(temporal_sample.frames):

    print("=" * 60)
    print(i)

    print(frame.events_t.dtype)
    print(frame.events_t.shape)

    print(frame.events_t[:10])

    print(frame.events_t.min())
    print(frame.events_t.max())

In [ ]:
loader = DataLoader(
    temporal_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

batch = next(iter(loader))
frame = batch.frames[0]

print(type(frame.events_t))
print(frame.events_t.dtype)
print(frame.events_t.shape)

print(frame.events_t[:10])

print(frame.events_t.min())
print(frame.events_t.max())

In [ ]:
print("=" * 80)
print("SEQUENCE INDEX")
print("=" * 80)

print("Sequences:", len(frame_dataset.index.sequences))
print("Frames   :", len(frame_dataset))

for seq in frame_dataset.index.sequences[:5]:
    print(
        seq.sequence_name,
        seq.sensor,
        seq.num_frames,
    )


print("=" * 80)
print("PARSER")
print("=" * 80)

parser = frame_dataset.parsers[0]

print(len(parser.frames))
print(len(parser.objects))

frame = parser.frames[0]

print(frame.frame_id)
print(frame.timestamp)

print(frame.depth_available)
print(frame.mask_available)

print(len(frame.object_states))

print("=" * 80)
print("READER")
print("=" * 80)

reader = frame_dataset.readers[0]

print(reader.events_xy.shape)
print(reader.events_t.shape)
print(reader.events_p.shape)

print(reader.events_t.min())
print(reader.events_t.max())

print(reader.events_xy.dtype)
print(reader.events_t.dtype)
print(reader.events_p.dtype)

print("=" * 80)
print("EVENT INDEX")
print("=" * 80)

cache = frame_dataset.event_indices[0]

print(cache.event_start.shape)
print(cache.event_end.shape)

for i in range(5):

    s = cache.event_start[i]
    e = cache.event_end[i]

    print(
        i,
        s,
        e,
        e-s,
    )

print("=" * 80)
print("IMU INDEX")
print("=" * 80)

cache = frame_dataset.imu_indices[0]

print(cache.imu_start.shape)
print(cache.imu_end.shape)

for i in range(5):

    s = cache.imu_start[i]
    e = cache.imu_end[i]

    print(
        i,
        s,
        e,
        e-s,
    )

In [ ]:
print("=" * 80)
print("DATASET SAMPLE")
print("=" * 80)

sample = frame_dataset[100]

print(sample.sequence_name)
print(sample.sensor)
print(sample.frame_id)

print(sample.events_xy.shape)
print(sample.events_t.shape)
print(sample.events_p.shape)

print(sample.imu.timestamps.shape)
print(sample.imu.angular_velocity.shape)
print(sample.imu.linear_acceleration.shape)

print(sample.depth.shape if sample.depth is not None else None)
print(sample.mask.shape if sample.mask is not None else None)
print(sample.rgb.shape if sample.rgb is not None else None)


print("=" * 80)
print("TEMPORAL DATASET")
print("=" * 80)

sample = temporal_dataset[50]

print(sample.history_offsets)

for i, frame in enumerate(sample.frames):

    print(
        i,
        frame.frame_id,
        frame.timestamp,
        frame.events_t.shape[0],
        frame.imu.timestamps.shape[0],
    )

loader = DataLoader(
    temporal_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

batch = next(iter(loader))

print("=" * 80)
print("COLLATE")
print("=" * 80)

print(len(batch.frames))

for i, frame in enumerate(batch.frames):

    print(
        i,
        frame.events_xy.shape,
        frame.events_t.shape,
        frame.event_sample_indices.shape,
        frame.imu.timestamps.shape,
        frame.imu.sample_indices.shape,
    )


tensor_batch = pipeline(batch)

print("=" * 80)
print("TENSOR")
print("=" * 80)

frame = tensor_batch.frames[0]

print(frame.events_xy.dtype)
print(frame.events_t.dtype)

print(frame.imu.timestamps.dtype)
print(frame.imu.angular_velocity.dtype)


for sid in torch.unique(frame.event_sample_indices):

    mask = frame.event_sample_indices == sid

    t = frame.events_t[mask]

    print(
        sid.item(),
        t.min().item(),
        t.max().item(),
    )

for sid in torch.unique(frame.imu.sample_indices):

    mask = frame.imu.sample_indices == sid

    t = frame.imu.timestamps[mask]

    print(
        sid.item(),
        t.min().item(),
        t.max().item(),
    )


motion = frame.camera_motion[0]

print(motion.translation)
print(motion.delta_translation)

print(motion.linear_speed)
print(motion.angular_speed)


obj = frame.frame_motion[0]

print(obj.object_ids.shape)
print(obj.delta_position.shape)
print(obj.speed.shape)

print(obj.speed.min())
print(obj.speed.max())


batch = next(iter(loader))

original = batch.frames[0].events_t.copy()

_ = pipeline(batch)

assert np.allclose(
    original,
    batch.frames[0].events_t,
)

print("✓ No shared memory corruption.")

In [ ]:
for t, frame in enumerate(batch.frames):

    print("=" * 80)
    print(f"Temporal frame {t}")

    print("\nEvents")
    print("Total events:", len(frame.events_t))

    print("\nPer-sample")

    for sample_id, imu in enumerate(frame.imu):

        event_mask = frame.event_sample_indices == sample_id

        event_count = int(event_mask.sum())

        print(
            f"Sample {sample_id}:",
            f"Events={event_count:6d}",
            f"IMU={len(imu.timestamps):2d}",
            f"Event [{frame.events_t[event_mask][0]:.6f}, {frame.events_t[event_mask][-1]:.6f}]",
            f"IMU [{imu.timestamps[0]:.6f}, {imu.timestamps[-1]:.6f}]",
        )

In [ ]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)


from pathlib import Path

from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn
from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
)

# ==========================================================
# DATASET
# ==========================================================

dataset_root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

frame_dataset = EVIMO2Dataset(
    dataset_root=dataset_root,
    sensors=("left_camera", "right_camera"),
    split="train",
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)

loader = DataLoader(
    temporal_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

batch = next(iter(loader))

# ==========================================================
# PRE-TENSOR CHECK
# ==========================================================

print("=" * 80)
print("RAW BATCH")
print("=" * 80)

for t, frame in enumerate(batch.frames):

    print(f"\nTemporal frame {t}")

    print(
        "Events:",
        frame.events_xy.shape,
        frame.events_t.shape,
        frame.events_p.shape,
    )

    print(
        "IMU:",
        frame.imu_timestamps.shape,
        frame.imu_angular_velocity.shape,
        frame.imu_linear_acceleration.shape,
    )

    print()

    for sample_id in range(len(frame.sequence_names)):

        e = frame.event_sample_indices == sample_id
        i = frame.imu_sample_indices == sample_id

        et = frame.events_t[e]
        it = frame.imu_timestamps[i]

        print(
            f"Sample {sample_id:2d} | "
            f"Events={len(et):6d} | "
            f"IMU={len(it):3d}"
        )

        #
        # Event timestamps must be increasing
        #
        assert (et[1:] >= et[:-1]).all()

        #
        # IMU timestamps must be increasing
        #
        assert (it[1:] >= it[:-1]).all()

        #
        # IMU should lie inside event window
        #
        if len(et) and len(it):

            assert it[0] >= et[0] - 0.002
            assert it[-1] <= et[-1] + 0.002

print("\n✓ Raw batch verified")

# ==========================================================
# TRANSFORMS
# ==========================================================

transform = Compose([
    ToTensor(),
    NormalizeEventTime(),
    NormalizeIMU(),
])

tensor_batch = transform(batch)

print("\n")
print("=" * 80)
print("TENSOR BATCH")
print("=" * 80)

for t, frame in enumerate(tensor_batch.frames):

    print(f"\nTemporal frame {t}")

    #
    # Types
    #
    assert frame.events_xy.dtype.is_floating_point is False
    assert frame.events_t.dtype == frame.imu_timestamps.dtype
    assert frame.events_p.dtype == frame.imu_angular_velocity.dtype

    #
    # Shapes
    #
    assert frame.events_xy.shape[0] == frame.events_t.shape[0]
    assert frame.events_xy.shape[0] == frame.events_p.shape[0]

    assert frame.imu_timestamps.shape[0] == frame.imu_angular_velocity.shape[0]
    assert frame.imu_timestamps.shape[0] == frame.imu_linear_acceleration.shape[0]

    #
    # Per sample
    #
    sample_ids = frame.event_sample_indices.unique()

    for sid in sample_ids:

        sid = int(sid)

        e = frame.event_sample_indices == sid
        i = frame.imu_sample_indices == sid

        et = frame.events_t[e]
        it = frame.imu_timestamps[i]

        #
        # Event normalization
        #
        if len(et):

            assert abs(float(et[0])) < 1e-6
            assert abs(float(et[-1]) - 1.0) < 1e-6

        #
        # IMU normalization
        #
        if len(it) >= 2:

            assert abs(float(it[0])) < 1e-6
            assert abs(float(it[-1]) - 1.0) < 1e-6

        elif len(it) == 1:

            assert abs(float(it[0])) < 1e-6

print("\n✓ Tensor batch verified")

print("\n")
print("=" * 80)
print("ALL TESTS PASSED")
print("=" * 80)

In [ ]:

from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

# ==========================================================
# Dataset
# ==========================================================

dataset_root = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

frame_dataset = EVIMO2Dataset(
    dataset_root=dataset_root,
    sensors=("left_camera", "right_camera"),
    split="train",
    load_depth=True,
    load_mask=True,
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=(-3, -2, -1, 0),
)

loader = DataLoader(
    temporal_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)

# ==========================================================
# Raw batch
# ==========================================================

raw_batch = next(iter(loader))

print("=" * 90)
print("RAW BATCH")
print("=" * 90)

for t, frame in enumerate(raw_batch.frames):

    print(f"\nTemporal frame {t}")

    print("Events")
    print("  xy :", frame.events_xy.shape)
    print("  t  :", frame.events_t.shape)
    print("  p  :", frame.events_p.shape)

    print()

    print("IMU")
    print("  timestamps :", frame.imu_timestamps.shape)
    print("  gyro       :", frame.imu_angular_velocity.shape)
    print("  accel      :", frame.imu_linear_acceleration.shape)

print()

# ==========================================================
# Transform pipeline
# ==========================================================

transform = Compose(

    [

        ToTensor(),

        NormalizeEventTime(),

        NormalizeIMU(),

        VoxelizeEvents(
            num_bins=5,
        ),

    ]

)

voxel_batch = transform(raw_batch)

# ==========================================================
# Voxel batch
# ==========================================================

print("=" * 90)
print("VOXEL BATCH")
print("=" * 90)

assert len(voxel_batch.frames) == len(
    raw_batch.frames
)

for temporal_index, frame in enumerate(voxel_batch.frames):

    print()

    print("=" * 70)
    print(f"Temporal frame {temporal_index}")
    print("=" * 70)

    voxel = frame.voxel_grid

    print()

    print("Voxel grid")
    print("Shape :", tuple(voxel.shape))
    print("dtype :", voxel.dtype)

    #
    # Expected shape
    #
    assert voxel.ndim == 4

    batch_size = len(frame.metadata.sequence_names)

    assert voxel.shape[0] == batch_size
    assert voxel.shape[1] == 5
    assert voxel.shape[2] == 480
    assert voxel.shape[3] == 640

    #
    # Values
    #
    print()

    print("Statistics")

    print("Min :", voxel.min().item())
    print("Max :", voxel.max().item())

    abs_sum = voxel.abs().sum().item()

    print("Absolute Sum :", abs_sum)

    assert abs_sum > 0

    #
    # Every sample should contain events
    #
    print()

    print("Per sample magnitude")

    for sample_index in range(batch_size):

        magnitude = voxel[sample_index].abs().sum().item()

        print(
            f"Sample {sample_index:2d}: {magnitude:.3f}"
        )

        assert magnitude > 0

    #
    # Metadata consistency
    #
    metadata = frame.metadata

    print()

    print("Metadata")

    print("Batch size :", len(metadata.sequence_names))

    assert len(metadata.sequence_names) == batch_size
    assert len(metadata.camera_motion) == batch_size
    assert len(metadata.frame_motion) == batch_size
    assert len(metadata.depth) == batch_size
    assert len(metadata.mask) == batch_size
    assert len(metadata.rgb) == batch_size

print()

print("=" * 90)
print("ALL VOXELIZATION TESTS PASSED")
print("=" * 90)

In [ ]:
sample = frame_dataset[0]

print("="*80)
print("CAMERA CALIBRATION TEST")
print("="*80)

print()

print("K:")
print(sample.camera_intrinsics)

print()

print("D:")
print(sample.camera_distortion)

print()

print("Shapes")
print(
    "K:",
    sample.camera_intrinsics.shape
)

print(
    "D:",
    sample.camera_distortion.shape
)